#### Imports and setup

This cell imports helper functions and creates:
- a gold_run_id value
- a run date string
- a run timestamp string

These are used for tracking and snapshot publishing

In [0]:
%run "../notebooks/helper_functions"

In [0]:
# Create widgets for configuration
dbutils.widgets.text("catalog_name", "novacart_catalog", "1. Target Catalog Name")
dbutils.widgets.text("silver_schema", "silver_schema", "2. Silver Schema Name")
dbutils.widgets.text("gold_schema", "gold_schema", "3. Gold Schema Name")

# Read widget values
catalog_name = dbutils.widgets.get("catalog_name").strip()
silver_schema = dbutils.widgets.get("silver_schema").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()

# Construct table prefixes and control table
silver_prefix = f"{catalog_name}.{silver_schema}"
gold_prefix = f"{catalog_name}.{gold_schema}"
gold_control_table = f"{catalog_name}.{gold_schema}.processing_control"

print(f"Silver Prefix: {silver_prefix}")
print(f"Gold Prefix: {gold_prefix}")
print(f"Gold Control Table: {gold_control_table}")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
# Try to get run_id from job context (works when running as a job)
try:
  run_id = spark.conf.get("spark.databricks.runId")
  gold_run_id = run_id
except Exception:
  # Not running as a job, generate a UUID
  gold_run_id = str(uuid.uuid4())
  print(f"Running interactively, generated UUID: {gold_run_id}")

run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print(f"Current Gold Run ID: {gold_run_id}")
print(f"Run timestamp Folder: {run_ts_str}")
print(f"Run date: {run_date_str}")

#### Read changed Silver rows only

This cell reads the full Silver current-state tables but filters only the rows that changed since the last Gold run

This is the starting poing for Gold incremental processing.

In [0]:
# Pass control_table parameter
last_gold_ts = get_last_processed_silver_ts("orders_information", gold_control_table)

print("Last Processed Silver Timestamp for Gold=", last_gold_ts)

silver_orders_current = spark.read.table(f"{silver_prefix}.orders_transformed")
silver_products_current = spark.read.table(f"{silver_prefix}.products_transformed")
silver_payments_current = spark.read.table(f"{silver_prefix}.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    changed_orders = silver_orders_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(F.col("processed_at") > F.lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed orders = {changed_orders_count}")
print(f"Number of changed products = {changed_products_count}")
print(f"Number of changed payments = {changed_payments_count}")

#### Find impacted order IDs
Gold is built at order grain, so if anything changes in orders, products or payments, we identify which order_id values are impacted.
Only those order IDs are rebuilt in Gold.

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"),
          F.col("p.product_id") == F.col("o.product_id"),
          "inner")
    .select(F.col("o.order_id")).distinct()
)

impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_orders)
    .distinct()
)

print(f"impacted order ids = {impacted_order_ids.count()}")
display(impacted_order_ids.orderBy("order_id"))

#### Build Gold delta for impacted orders
This cell joins the impacted orders with the current Silver products and payments tables, derives business columns, and builds the **Gold delta** that will be merged into the Gold current-state table.

In [0]:
impacted_order = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_order.alias("o")
    .join(
        silver_products_current.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("py"),
        F.col("o.order_id") == F.col("py.order_id"),
        "inner"
    )
    .withColumn(
        "payment_completion_ratio",
        F.when(F.col("o.order_amount") == 0, F.lit(0.0))
        .otherwise(F.col("py.paid_amount") / F.col("o.order_amount"))
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("product_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("py.payment_id"),
        F.col("py.payment_status"),
        F.col("py.paid_amount"),
        F.col("payment_completion_ratio"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("py.processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_state",
        F.when(F.col("order_amount") == 0, "Invalid_order_amount")
        .when(F.col("payment_completion_ratio") == 0, "Unpaid")
        .when(F.col("payment_completion_ratio") == 1, "Paid")
        .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
        .when(F.col("payment_completion_ratio") > 1, "Overpaid")
        .otherwise("Unknown")
    )
    .withColumn("gold_updated_date", F.to_date(F.col("gold_update_ts")))
    .withColumn("gold_run_id", F.lit(gold_run_id))
)

print("Gold delta rows =", gold_delta.count())
display(gold_delta)

#### Merge Gold current-state table
If Gold delta contains rows, this cell merges them into gold_schema.orders_information. If there are no impacted rows, nothing is merged

In [0]:
if gold_delta.count() > 0:
    # Pass control_table parameter (not needed for upsert_to_gold)
    upsert_to_gold(gold_delta, f"{gold_prefix}.orders_information", "order_id")
else:
    print("No new rows to insert in gold table")

In [0]:
display_sql_stmt = f"SELECT * FROM {gold_prefix}.orders_information"
orders_info_df = spark.sql(display_sql_stmt)
display(orders_info_df)

In [0]:
if not spark.catalog.tableExists(f"{gold_prefix}.orders_information_scd2"):
    spark.sql(f"""
    CREATE TABLE {gold_prefix}.orders_information_scd2
    USING delta AS
    SELECT *, 
           CAST(NULL AS timestamp) AS valid_from_ts,
           CAST(NULL AS timestamp) AS valid_to_ts,
           true AS is_current
    FROM {gold_prefix}.orders_information
    WHERE 1 = 0
    """)

if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    
    spark.sql(f"""
    MERGE INTO {gold_prefix}.orders_information_scd2 t
    USING gold_delta_view s
    ON t.order_id = s.order_id AND t.is_current = true
    WHEN MATCHED AND (
        NOT(t.order_status <=> s.order_status) OR 
        NOT(t.order_amount <=> s.order_amount) OR 
        NOT(t.paid_amount <=> s.paid_amount) OR 
        NOT(t.payment_id <=> s.payment_id) OR 
        NOT(t.category <=> s.category) OR 
        NOT(t.product_name <=> s.product_name) OR 
        NOT(t.product_price <=> s.product_price) )
    THEN UPDATE SET 
        is_current = false,
        valid_to_ts = s.gold_update_ts
    """)

    spark.sql(f"""
    INSERT INTO {gold_prefix}.orders_information_scd2
    SELECT s.*,
           s.gold_update_ts AS valid_from_ts,
           CAST(NULL AS timestamp) AS valid_to_ts,
           true AS is_current
    FROM gold_delta_view s 
    LEFT JOIN {gold_prefix}.orders_information_scd2 t 
    ON s.order_id = t.order_id AND t.is_current = true 
    WHERE t.order_id IS NULL OR (
        NOT(t.order_status <=> s.order_status) OR 
        NOT(t.order_amount <=> s.order_amount) OR 
        NOT(t.paid_amount <=> s.paid_amount) OR
        NOT(t.payment_id <=> s.payment_id) OR
        NOT(t.category <=> s.category) OR
        NOT(t.product_name <=> s.product_name) OR
        NOT(t.product_price <=> s.product_price)
    )
    """)
    
    print(f"Updated SCD2 table: {gold_prefix}.orders_information_scd2")

#### Update category-level Gold aggregation

This cell recalculates category-level business metrics only for categories impacted in the current run, then merges them into the category performance Gold table

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta
        .select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )

    category_perf_delta = (
        spark.read.table(f"{gold_prefix}.orders_information")
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount"))
                .otherwise(F.lit(0.0))
            ).alias("gross_merchandise_value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                .otherwise(F.lit(0.0))
            ).alias("total_paid_amount"),
            F.avg(F.col("payment_completion_ratio")).alias("avg_payment_completion_ratio"),
            ( F.sum(
                F.when(F.col("payment_status") == "FAILED", 1).otherwise(0)
            ) / F.count("*") ).alias("payment_failure_rate")
        )
    )
    
    upsert_to_gold(category_perf_delta, f"{gold_prefix}.category_performance", "category")
    print(f"Updated category performance table: {gold_prefix}.category_performance")
else:
    print("No new rows to insert in gold table")

In [0]:
display_sql_stmt = f"SELECT * FROM {gold_prefix}.category_performance"
cat_perf_df = spark.sql(display_sql_stmt)
display(cat_perf_df)

#### Publish Gold snapshots to Volumne
This cell writes two kinds of Gold outputs to a Databricks Volume
- **latest snapshot** - overwritten every successful run
- **timestamped historical snapshot** - a new folder for each successful run

This is useful for audit, rollback, and teaching demos. 

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {gold_prefix}.gold_snapshots_vol")
print(f"Volume created/verified: {gold_prefix}.gold_snapshots_vol")

In [0]:
# Latest snapshot paths (overwritten each run)
latest_orders_path = f"/Volumes/{catalog_name}/{gold_schema}/gold_snapshots_vol/gold_latest/orders_information"
latest_category_path = f"/Volumes/{catalog_name}/{gold_schema}/gold_snapshots_vol/gold_latest/category_performance"

# Historical snapshot paths (timestamped folders)
historical_orders_path = f"/Volumes/{catalog_name}/{gold_schema}/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"
historical_category_path = f"/Volumes/{catalog_name}/{gold_schema}/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"

# Write latest snapshots (overwrite mode)
spark.read.table(f"{gold_prefix}.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)
spark.read.table(f"{gold_prefix}.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

# Write historical snapshots (append mode with partitions)
spark.read.table(f"{gold_prefix}.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table(f"{gold_prefix}.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print(f"Latest Orders Path: {latest_orders_path}")
print(f"Latest Category Path: {latest_category_path}")
print(f"Historical Orders Path: {historical_orders_path}")
print(f"Historical Category Path: {historical_category_path}")

#### Update Gold Control Table
This final cell updates the Gold COntrol table using the latest Silver processing metadata and displays the control table for validation

In [0]:
latest_silver_ts = silver_orders_current.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None

# Pass control_table parameter
upsert_gold_control(
    "orders_information",
    latest_silver_run_id,
    latest_silver_ts,
    gold_delta.count(),
    gold_run_id,
    gold_control_table
)

print(f"Updated gold control table: {gold_control_table}")
display(spark.table(gold_control_table))